# 3D balanced SSFP (bSSFP)

**Balanced SSFP** is a gradient-echo family defined by a condition on its gradients: over one
repetition -- from one RF pulse to the next -- the net gradient area on **every** axis returns to
zero. Each repetition excites, encodes one line of k-space, reads it out, and unwinds all of its
encoding again, leaving nothing behind on any axis.

That condition is what the sequence is named for, and it is all a single repetition can promise.
Run many of them at a short, constant TR and the transverse magnetisation surviving each pulse can
stay coherent from one repetition to the next instead of being spoiled away, which is what allows a
bSSFP train to settle into its characteristic high-signal steady state. **Whether it actually
does** depends on T1, T2, the flip angle, the RF phase schedule and the off-resonance distribution
-- none of which a single repetition knows anything about. Balance removes the net gradient
dephasing of stationary spins; it does not by itself keep everything in phase.

bSSFP is used where signal per unit time matters: cardiac cine, fetal imaging, fast abdominal work.
Its price is a sensitivity to off-resonance that appears as dark **banding** wherever the field
drifts far enough across a repetition.

This notebook builds the **3D** version, where a second phase-encode axis divides a slab into
partitions. The 2D sequence gives `z` one job -- select a slice and undo that selection. In 3D the
same axis has to do three: select the slab, balance that selection, **and** encode `kz`. How those
three share one axis is the question here, and the answer turns out to be about *ownership*
between neighbouring repetitions rather than about arithmetic.

In [ ]:
from pathlib import Path

import numpy as np
import pypulseq as pp

import seqcraft as sc

opts = pp.Opts(
    max_grad=32, grad_unit='mT/m',
    max_slew=130, slew_unit='T/m/s',
    B0=3.0,
    rf_dead_time=100e-6,
    rf_ringdown_time=30e-6,
    adc_dead_time=10e-6,
)

FOV_MM = (200.0, 200.0, 160.0)      # x, y, and the slab extent the partitions encode
MATRIX = (64, 64, 16)               # readout samples, phase-encode lines, partitions
SLAB_MM = 100.0                     # the excited slab -- narrower than the encoded z extent
FLIP_DEG = 35.0                     # bSSFP runs hot: the steady state wants a large angle
BANDWIDTH_HZ_PX = 400.0             # wide, because a short TR is the whole point

NX, NY, NZ = MATRIX
SEQ_DIR = Path('seq')
SEQ_DIR.mkdir(exist_ok=True)

print(f'{NX} x {NY} x {NZ} over {FOV_MM[0]:.0f} x {FOV_MM[1]:.0f} x {FOV_MM[2]:.0f} mm')
print(f'voxel {FOV_MM[0] / NX:.2f} x {FOV_MM[1] / NY:.2f} x {FOV_MM[2] / NZ:.2f} mm')
print(f'excited slab {SLAB_MM:.0f} mm inside an encoded z extent of {FOV_MM[2]:.0f} mm')

### Two different z numbers

`SLAB_MM` and `FOV_MM[2]` are independent, and here they deliberately differ. The slab thickness is
an RF and gradient property: how much of the patient the pulse excites. The encoded z extent is a
k-space sampling property: how far apart the partition steps are, and therefore how wide a region
is unambiguously resolved.

Encoding a wider extent than is excited is the z equivalent of oversampling. A slab profile is not
a perfect rectangle, and giving the encoding room on either side of it keeps the profile's soft
edges from wrapping into the reconstructed volume. The alternative -- matching the two -- puts the
imperfect edges of the excitation directly at the edges of the encoded volume.

## One repetition

`sc.modules.bSSFP3DTR` is the repeating unit: one excitation, one encoded readout, and the
gradient lobes that return every axis to zero. It is **not** a scan — it does not know how many
repetitions there are, what order they run in, or how the steady state is established. Those
belong to whatever writes the loop, and the last section comes back to why.

In [ ]:
tr = sc.modules.bSSFP3DTR(
    opts=opts,
    fov_mm=FOV_MM,
    matrix=MATRIX,
    slab_thickness_mm=SLAB_MM,
    flip_deg=FLIP_DEG,
    bandwidth_hz_px=BANDWIDTH_HZ_PX,
)

print(f'TE               {tr.te_s * 1e3:7.3f} ms')
print(f'TR               {tr.tr_s * 1e3:7.3f} ms')
print(f'TE / TR          {tr.te_s / tr.tr_s:7.4f}')
print(f'centre of k      line {tr.center_line}, partition {tr.center_partition}')
print(f'dkz              {tr.dk_per_m("z"):7.3f} 1/m')
print(f'symmetry residual{tr.symmetry_residual_s * 1e6:7.3f} us '
      f'(raster is {float(opts.grad_raster_time) * 1e6:.0f} us)')

### TE = TR/2 is the realisation, not the definition

Three things are worth keeping apart, because a notebook that collapses them teaches the wrong
causality:

```text
the balance definition        net gradient area zero on every axis, RF centre to RF centre
the default realisation       TE = TR/2, the symmetric placement of the echo in that interval
the steady-state contrast     a property of the repeated train: tissue relaxation, the flip
                              angle, the RF phase schedule and the off-resonance response
```

Balance says nothing about where the echo falls; an asymmetric echo is still balanced, and this
module will build one when asked. Putting the echo at the midpoint is the conventional symmetric
realisation and is the default here. It is not what *creates* bSSFP's contrast -- that belongs to
the train, not to a single repetition's echo placement.

`symmetry_residual_s` above is how far the echo sits from the exact midpoint. It is not zero
because the midpoint need not land on the gradient raster, and the module reports what it achieved
rather than rounding the TE it declares.

## The timing origin is the RF effective centre

Every time this module declares is measured from the **effective centre** of the RF pulse, read
from the waveform rather than taken as the midpoint of the event:

```text
TE               = echo - RF effective centre of repetition n
TR               = RF effective centre of n+1 - RF effective centre of n
balance interval = [RF effective centre of n, RF effective centre of n+1]
```

For a symmetric pulse the effective centre is near the middle; for a minimum-phase pulse it is
nowhere near it, and a caller who derived the origin from a pulse duration would be declaring a TE
it does not achieve. `time_to_rf_center()` reports where it lands inside the block so nothing
downstream has to reconstruct it.

In [ ]:
print(f'block starts at              0.000 ms')
print(f'RF effective centre at   {tr.time_to_rf_center() * 1e3:9.3f} ms')
print(f'echo at                  {tr.time_to_echo() * 1e3:9.3f} ms')
print(f'block ends at            {tr.build(line=0, partition=0).duration * 1e3:9.3f} ms')
print()
print(f'echo - RF centre         {(tr.time_to_echo() - tr.time_to_rf_center()) * 1e3:9.3f} ms'
      f'   = TE ({tr.te_s * 1e3:.3f} ms)')

## The z axis does three jobs

The slab-selection gradient is playing while the RF is on, so part of its area falls **before** the
RF effective centre and part **after** it. Call those `A_pre` and `A_post`. The module measures
both from the waveform rather than assuming they are equal: for a symmetric pulse they nearly are,
but for a minimum-phase pulse the effective centre is nowhere near the waveform's midpoint, and
assuming equality there would mis-state the interval the balance condition is about.

Over one RF-to-RF interval the z axis therefore sees contributions from **two** repetitions, plus
whatever encodes the partition. This module plays three z lobes per repetition:

```text
lead(n)   = -A_pre(n)             before the pulse    -- cancels this pulse's leading portion
tail(n)   = -A_post(n) + K(p)     after the pulse     -- cancels its trailing portion AND encodes kz
rewind(n) = -K(p)                 after the echo      -- unwinds this repetition's own encoding
```

`K(p)` is the k-space position partition `p` encodes, which is `pe_z.k_per_m(p)`. The encode is
**lumped into** the balancing lobe rather than played beside it: the z axis is already busy between
the pulse and the echo, and a second gradient there would merge with the first into a waveform that
exceeds the slew limit at the edge of the partition table. One lobe carrying the signed sum is the
same moment in a legal shape.

The rest of this notebook does not inspect those lobes directly. It measures what they produce --
the net area over the interval, and the k-space position at the echo -- which is the thing the
contract is actually about.

In [ ]:
print(f'{"partition":>10}{"K(p), 1/m":>12}{"":>4}{"of the table":>14}')
for p in (0, 1, tr.center_partition, NZ - 2, NZ - 1):
    k = tr.pe_z.k_per_m(p)
    marker = '  <- centre, encodes nothing' if p == tr.center_partition else ''
    print(f'{p:>10}{k:>12.2f}{"":>4}{k / tr.dk_per_m("z"):>+10.0f} dkz{marker}')
print()
print(f'dkz = {tr.dk_per_m("z"):.3f} 1/m, so the table spans '
      f'{tr.pe_z.k_per_m(0):.1f} .. {tr.pe_z.k_per_m(NZ - 1):+.1f} 1/m')

### Why one window for the whole table

Every partition uses the **same** lobe duration. Letting each take its own shortest would make TE
a function of `kz` — a contrast gradient across the volume that no k-space check would reveal and
no reconstruction expects. So the window is sized by whichever partition needs the longest, and
every other partition plays a stretched lobe of its own area.

Which partition that is comes out of the *signed* combination `-A_post + K(p)`, not from the index:
reverse the slab gradient and the limit moves to the other edge of k-space. Every supported
partition is enumerated rather than an edge assumed.

The leading lobe takes that same window, and that is what keeps the z structure symmetric about
the echo -- which the first-moment section below measures.

## Balance is a condition on an interval, not on a block

The interval runs from one RF effective centre to the next, so it **ends inside the following
repetition**. Measuring a single block from its first event to its last is a different quantity,
and it can read zero for a design that is not balanced at all.

The function below integrates the compiled waveforms between consecutive RF centres, which is
where the condition is actually defined.

In [ ]:
def rf_centres(seq):
    '''Every RF effective centre in a compiled sequence, seconds from the start.'''
    t, out = 0.0, []
    for i in range(1, len(seq.block_events) + 1):
        block = seq.get_block(i)
        rf = getattr(block, 'rf', None)
        if rf is not None:
            centre, _ = pp.calc_rf_center(rf)
            out.append(t + float(rf.delay) + centre)
        t += float(seq.block_durations[i])
    return out


def m0_between(seq, a, b):
    '''Net gradient area on x, y, z over [a, b], 1/m.'''
    out = []
    for axis in range(3):
        times, amps = seq.waveforms_and_times()[0][axis]
        t, g = np.asarray(times, float), np.asarray(amps, float)
        if t.size == 0:
            out.append(0.0)
            continue
        grid = np.unique(np.concatenate([t[(t > a) & (t < b)], [a, b]]))
        out.append(float(np.trapezoid(np.interp(grid, t, g, left=0.0, right=0.0), grid)))
    return np.array(out)


def train(views):
    '''A compiled train of (line, partition) views, stacked at the declared TR.'''
    tree, t = sc.LogicBlock('bssfp_3d'), 0.0
    for line, partition in views:
        tree.add(t, tr.build(line=line, partition=partition))
        t += tr.tr_s
    return sc.compile(tree, opts)

### Each repetition must discharge its own encoding

Here is the part that is specific to 3D, and it is a question about *ownership* rather than about
arithmetic.

Suppose the encode went into the lobe before the echo and the rewind into the lobe before the
**next** pulse. Every individual repetition would still look tidy, and a train in which every
repetition used the same partition would balance perfectly. But the interval spans two
repetitions, so what it actually carries is

```text
A_post(n) + [-A_post + K(p_n)] + [-A_pre - K(p_{n+1})] + A_pre(n+1)  =  K(p_n) - K(p_{n+1})
```

which is zero **only while neighbouring repetitions share a partition**. A 3D acquisition chooses
its own view order, so a repetition whose balance depends on what its successor happens to acquire
is not a usable repetition.

Giving the encode and its rewind to the *same* repetition removes the coupling. The test is a view
order where every neighbouring pair changes **both** indices — a train of identical repetitions
cannot tell the two designs apart.

In [ ]:
views = [(tr.center_line, 0), (0, NZ - 1), (NY - 1, tr.center_partition), (1, 1),
         (tr.center_line, NZ - 1), (NY - 1, 0), (32, NZ - 2)]
seq = train(views)
centres = rf_centres(seq)

print(f'{"from":>12}{"to":>12}{"d kz":>9}{"M0x":>11}{"M0y":>11}{"M0z":>11}')
worst = 0.0
for k in range(len(centres) - 1):
    m0 = m0_between(seq, centres[k], centres[k + 1])
    worst = max(worst, np.abs(m0).max())
    dk = tr.pe_z.k_per_m(views[k + 1][1]) - tr.pe_z.k_per_m(views[k][1])
    print(f'{str(views[k]):>12}{str(views[k + 1]):>12}{dk:>9.2f}'
          f'{m0[0]:>11.1e}{m0[1]:>11.1e}{m0[2]:>11.1e}')
print()
print(f'worst |M0| over every interval and axis: {worst:.2e} 1/m')

Zero on all three axes across partition jumps spanning the whole table. The residual is float
accumulation over the train, not a property of the design.

## What was actually encoded

Balance alone is not correctness. A repetition that returns every axis to zero but rewinds the
*wrong* partition is still wrong, and its k-space trace would look perfectly tidy. So the echo of
each repetition is checked against the `(ky, kz)` it was asked for.

In [ ]:
k_adc = seq.calculate_kspace()[0]
samples = tr.ro.adc.num_samples
per_rep = k_adc.reshape(3, -1, samples)
echo = tr.ro.echo_sample(0)

print(f'{"view":>12}{"kx":>10}{"ky":>10}{"kz":>10}{"want ky":>10}{"want kz":>10}')
for i, (line, partition) in enumerate(views):
    kx, ky, kz = per_rep[:, i, echo]
    print(f'{str((line, partition)):>12}{kx:>10.2f}{ky:>10.2f}{kz:>10.2f}'
          f'{tr.pe.k_per_m(line):>10.2f}{tr.pe_z.k_per_m(partition):>10.2f}')

## The waveform over one repetition

The z trace below shows the three lobes. Note that the two selection-side lobes sit the same
distance from their own RF centre and carry the same duration — that symmetry is a *choice* the
default realisation makes, not part of the definition of balance, and the next section measures
what it buys.

In [ ]:
import matplotlib.pyplot as plt

one = sc.compile(tr.build(line=tr.center_line, partition=NZ - 1), opts)
fig, axes = plt.subplots(3, 1, figsize=(9, 5.5), sharex=True)
for row, (axis, name) in enumerate(zip(range(3), ('x -- readout', 'y -- phase encode',
                                                  'z -- slab, balance, partition'))):
    times, amps = one.waveforms_and_times()[0][axis]
    axes[row].plot(np.asarray(times) * 1e3, np.asarray(amps) / 1e3, lw=1.4)
    axes[row].axhline(0.0, color='k', lw=0.6)
    axes[row].axvline(tr.time_to_echo() * 1e3, color='crimson', ls=':', lw=1.2)
    axes[row].axvline(tr.time_to_rf_center() * 1e3, color='tab:green', ls='--', lw=1.2)
    axes[row].set_ylabel(name, fontsize=8)
axes[0].set_title('one repetition at the edge of the partition table '
                  '(green = RF centre, red = echo)', fontsize=10)
axes[-1].set_xlabel('ms')
fig.tight_layout()

### What the symmetric realisation buys

At `kz = 0` the z axis carries nothing but slab-selection balancing, and because the two lobes are
equal and equally spaced about the echo, the **first** moment over the interval vanishes too — the
limit a symmetric bSSFP scheme is described by. Away from the centre the partition encoding adds
its own first-moment term, of opposite sign at the two edges of the table.

This is the same thing the phase encoding already does on `y`, and it is not a defect introduced by
going to 3D: an encode and its rewind sit on opposite sides of the echo, so their first moments add
rather than cancel.

In [ ]:
def m1z_about_echo(partition):
    '''First moment on z over one RF-to-RF interval, referenced to the echo.'''
    local = train([(tr.center_line, partition),
                   (0, (partition + 5) % NZ), (NY - 1, (partition + 9) % NZ)])
    c = rf_centres(local)
    times, amps = local.waveforms_and_times()[0][2]
    t, g = np.asarray(times, float), np.asarray(amps, float)
    echo_t = c[0] + tr.te_s
    grid = np.unique(np.concatenate([t[(t > c[0]) & (t < c[1])], [c[0], c[1]]]))
    y = np.interp(grid, t, g, left=0.0, right=0.0) * (grid - echo_t)
    return float(np.trapezoid(y, grid))


for p in (0, tr.center_partition, NZ - 1):
    print(f'partition {p:>2}  K = {tr.pe_z.k_per_m(p):+7.2f} 1/m   '
          f'M1z = {m1z_about_echo(p):+10.3e} s/m')

## Selective and non-selective are two excitation modes

`slab_thickness_mm=None` does not mean "the same design with the selection gradient removed". The
two modes have different RF families:

```text
slab-selective imaging default    shaped RF   + selection gradient
non-selective imaging default     hard RF     + no selection gradient
```

Those are two different physical excitations, and picking the right RF family is part of choosing
the mode rather than a consequence of it. The same rule is shared with `sc.modules.GRE3DTR`, so the
two 3D kernels cannot answer it differently.

A shaped but spatially non-selective pulse -- a spectrally selective excitation, for instance -- is
a third, deliberate design, and `rf_pulse` still asks for it. It is not what either default means.

In [ ]:
def excitation_shape(module):
    events = [ev for _, ev, _ in sc.flatten(module.exc(phase_deg=0.0))]
    rf = next(ev for ev in events if getattr(ev, 'type', None) == 'rf')
    channels = {ev.channel for ev in events if getattr(ev, 'type', None) in {'grad', 'trap'}}
    return np.asarray(rf.signal).size, float(pp.calc_duration(rf)), channels or {'none'}


common = dict(opts=opts, fov_mm=FOV_MM, matrix=MATRIX, flip_deg=FLIP_DEG,
              bandwidth_hz_px=BANDWIDTH_HZ_PX)
for label, module in (
    ('slab-selective', sc.modules.bSSFP3DTR(**common, slab_thickness_mm=SLAB_MM)),
    ('non-selective', sc.modules.bSSFP3DTR(**common, slab_thickness_mm=None)),
):
    n, duration, channels = excitation_shape(module)
    print(f'{label:<16} {n:>5} RF samples  {duration * 1e3:6.3f} ms  '
          f'gradient on {"/".join(sorted(channels))}')

## A volume

A complete acquisition is two loops over the repetition, plus start-up repetitions that load the
gradients without sampling. `acquire=False` drops the ADC and the labels and changes nothing else,
so a start-up repetition drives the magnetisation exactly as an acquired one does.

**The phase progression is the caller's.** Alternating the RF phase by 180 degrees each
repetition — a 0/pi phase cycle — is the common bSSFP convention and is what this loop does. The
RF phase increment sets the sequence's off-resonance frequency response, and so where the bands
fall; a different increment moves them. Which increment a protocol wants belongs to whatever knows
how many repetitions there are, not to a single repetition — the same reason RF spoiling lives
outside a spoiled gradient-echo kernel.

In [ ]:
STARTUP = 10                       # ramp repetitions, sampling nothing


def volume(kernel, *, startup=STARTUP):
    '''Every (partition, line) pair, with an alternating RF phase.'''
    tree, t, n = sc.LogicBlock('bssfp_3d'), 0.0, 0
    for _ in range(startup):
        tree.add(t, kernel.build(line=kernel.center_line, partition=kernel.center_partition,
                                 phase_deg=180.0 * (n % 2), acquire=False))
        t += kernel.tr_s
        n += 1
    for partition in range(NZ):
        for line in range(NY):
            tree.add(t, kernel.build(line=line, partition=partition,
                                     phase_deg=180.0 * (n % 2)))
            t += kernel.tr_s
            n += 1
    return tree, n


tree, repetitions = volume(tr)
seq_3d = sc.compile(tree, opts, name='bssfp_3d', definitions={
    'FOV': [FOV_MM[0] * 1e-3, FOV_MM[1] * 1e-3, FOV_MM[2] * 1e-3],
    'Matrix': [NX, NY, NZ],
    'TE': tr.te_s,
    'TR': tr.tr_s,
    'SlabThickness': SLAB_MM * 1e-3,
    'FlipAngle': FLIP_DEG,
})
print(f'{repetitions} repetitions ({STARTUP} start-up), '
      f'{repetitions * tr.tr_s:.2f} s of acquisition')

In [ ]:
seq_3d.write(str(SEQ_DIR / 'bssfp_3d.seq'))
np.savez(
    SEQ_DIR / 'bssfp_3d_nominal.npz',
    fov_mm=np.array(FOV_MM), matrix=np.array(MATRIX),
    slab_mm=SLAB_MM, flip_deg=FLIP_DEG,
    te_s=tr.te_s, tr_s=tr.tr_s, startup=STARTUP,
    center_line=tr.center_line, center_partition=tr.center_partition,
)
print(f'wrote {SEQ_DIR / "bssfp_3d.seq"}')

## The refusals

A request the design cannot reach is refused with the number that *is* reachable, rather than
silently rounded.

In [ ]:
for field, value in (('te_s', tr.min_te_s * 0.5),
                     ('tr_s', tr.min_tr_s * 0.5)):
    try:
        sc.modules.bSSFP3DTR(**common, slab_thickness_mm=SLAB_MM, **{field: value})
    except sc.ConfigurationError as error:
        print(f'{field}: {str(error).splitlines()[0]}')

try:
    tr.build(line=0, partition=NZ)
except sc.ConfigurationError as error:
    print(f'partition: {str(error).splitlines()[0]}')

try:
    sc.modules.bSSFP3DTR(**common, slab_thickness_mm=None, rf_time_bw_product=4.0)
except sc.ConfigurationError as error:
    print(f'rf_time_bw_product: {str(error).splitlines()[0]}')

## Limitations

**This builds a balanced waveform; it does not establish a steady state.** The two are different
claims. Zero net area per repetition is a property of the gradients and is checked above. Whether
the magnetisation has actually reached the bSSFP steady state depends on T1, T2, the flip angle and
how many repetitions have run — the ten start-up repetitions here are a placeholder, not a
calibrated preparation.

**Banding is not shown.** Off-resonance is where bSSFP actually fails, and seeing it needs a
simulation with a realistic field map rather than a waveform check.

**The partition encoding carries a first moment away from `kz = 0`**, measured above. For
stationary tissue that is irrelevant; for flow along the slab axis it is not, and this module does
not offer flow compensation on `z`.

**No reconstruction here.** The `.seq` file and the k-space trace are what this notebook produces.

## Summary

- `bSSFP3DTR` is one balanced repetition: net gradient area zero on every axis over the
  RF-centre-to-RF-centre interval, with `TE = TR/2` as the default realisation and not the
  definition.
- In 3D the `z` axis selects the slab, balances that selection, and encodes `kz`. The encode is
  lumped into the balancing lobe because the axis is already occupied in that window.
- **Each repetition discharges its own partition encoding**, so balance never depends on what the
  next repetition acquires — which is what lets the caller choose the view order. Measured across
  partition jumps spanning the whole table.
- One selection-side window serves every partition, which keeps TE and TR independent of `kz`.
  Both lobes take it, which gives the symmetric realisation whose first moment vanishes at
  `kz = 0`.
- Selective and non-selective are two excitation modes with different RF families, not one mode
  with a gradient switched off.
- The phase progression, start-up, segmentation and view order are the caller's.

## References

- Bieri O, Scheffler K. *Fundamentals of balanced steady state free precession MRI.*
  J Magn Reson Imaging 2013;38:2-11.
- Bernstein MA, King KF, Zhou XJ. *Handbook of MRI Pulse Sequences.* Elsevier, 2004.
  Section 14.1, "Balanced SSFP (True FISP)".
- Scheffler K, Lehnhardt S. *Principles and applications of balanced SSFP techniques.*
  Eur Radiol 2003;13:2409-2418.